# 🎬 Chapitrage & transcript des Conseils communaux — PV Explorer

Extrait, depuis la chaîne YouTube **@1030be**, le chapitrage (points débattus,
auteur·e, deep-link) et, optionnellement, le **contenu réel des débats**
(transcript aligné par point, via les sous-titres automatiques YouTube) des
séances filmées du Conseil communal de Schaerbeek.

Ce notebook télécharge et exécute **toujours la dernière version** des
scripts du dépôt (`pipeline/extract_video_chapters.py`,
`pipeline/prototype_transcript.py`, `pipeline/extract_transcripts.py`) —
rien n'est dupliqué ici, donc rien ne devient obsolète.

Quatre usages, indépendants (chacun nécessite ① et ②) :
- **③ Extraction complète du chapitrage** : scanne toute la chaîne.
- **④ Ajout manuel d'une séance** repérée à la main sur YouTube.
- **⑤ Test qualité du transcript** sur une seule séance (avant de généraliser).
- **⑥ Batch transcript** : enrichit toutes les séances déjà chapitrées d'une
  plage de dates avec le contenu réel des débats.


## ① Dépendances

In [ ]:
!pip install -U yt-dlp -q
!curl -fsSL https://deno.land/install.sh | DENO_INSTALL=/usr/local sh


## ② Charger le script de chapitrage (fonctions à jour du dépôt)

Télécharge et **définit** les fonctions du script (sans encore rien exécuter :
`__name__` est neutralisé pour que le scan complet ne se lance pas tout seul).

In [ ]:
import json
import types
import urllib.request

url = "https://raw.githubusercontent.com/pmeyssonnier/pv-explorer-app/main/pipeline/extract_video_chapters.py"
code_src = urllib.request.urlopen(url).read().decode("utf-8")

_mod = types.ModuleType("extract_video_chapters")
_mod.__dict__["__name__"] = "extract_video_chapters"   # neutralise `if __name__ == "__main__":`
exec(compile(code_src, "extract_video_chapters.py", "exec"), _mod.__dict__)
globals().update(_mod.__dict__)
print("Fonctions chargées :", "main, build_seance_entry, merge_seance, _fetch_video_info, ...")


## ③ Extraction complète du chapitrage (toutes les séances) — optionnel

In [ ]:
main()  # noqa: F821 -- chargé dynamiquement par la cellule ② (exec + globals().update())
# Le script lance le scan complet et écrit deux fichiers dans /content/ :
#  - pv_video_conseil_schaerbeek.json (chapitres + auteur·e·s + deep-links)
#  - video_sessions.json (date → URL de la vidéo de séance)


### Et ensuite ?

Télécharge les deux fichiers produits (panneau fichiers à gauche, ⋮ → *Download*)
et transmets-les pour intégration — **aucune réindexation Pinecone n'est
nécessaire**, ces fichiers sont lus directement par le backend.


## ④ Ajouter une séance repérée manuellement (optionnel)

Pour une vidéo que tu as trouvée toi-même sur YouTube (hors scan automatique :
titre non standard, playlist…), sans relancer tout le scan. Nécessite d'avoir
exécuté **① et ②** (pas besoin de ③).

Renseigne l'URL ci-dessous — la date est déduite du titre de la vidéo si
possible (« Conseil communal du JJ/MM/AAAA »), sinon précise-la explicitement
avec `date="AAAA-MM-JJ"`.

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=XXXXXXXXXXX"  # ← à remplacer
# DATE = "2026-07-15"   # décommente si la date n'est pas dans le titre

seance = build_seance_entry(VIDEO_URL)  # noqa: F821 -- idem, chargé par la cellule ②

# Fusionne avec les fichiers ACTUELS du dépôt (upsert par video_id — relancer
# sur la même vidéo la met simplement à jour, sans doublon).
base = "https://raw.githubusercontent.com/pmeyssonnier/pv-explorer-app/main/backend/"
chapitres = json.loads(urllib.request.urlopen(base + "video_conseil_schaerbeek.json").read())
sessions = json.loads(urllib.request.urlopen(base + "video_sessions.json").read())

chapitres["seances"] = merge_seance(chapitres["seances"], seance)  # noqa: F821
sessions[seance["date"]] = seance["video_url"]
sessions = dict(sorted(sessions.items(), reverse=True))

with open("/content/video_conseil_schaerbeek.json", "w", encoding="utf-8") as f:
    json.dump(chapitres, f, ensure_ascii=False, indent=2)
with open("/content/video_sessions.json", "w", encoding="utf-8") as f:
    json.dump(sessions, f, ensure_ascii=False, indent=2)

n_seances = len(chapitres["seances"])
print(f"✅ {n_seances} séances (dont celle ajoutée) → /content/video_conseil_schaerbeek.json")
print(f"✅ {len(sessions)} séances filmées → /content/video_sessions.json")


### Et ensuite ?

Télécharge les deux fichiers `/content/video_conseil_schaerbeek.json` et
`/content/video_sessions.json` et transmets-les pour intégration (remplacement
direct des fichiers du même nom dans `backend/`) — toujours **aucune
réindexation Pinecone requise** pour ces deux fichiers.


## ⑤ Test qualité du transcript (une séance) — optionnel

Vérifie, sur **une seule séance déjà chapitrée**, si des sous-titres
automatiques YouTube sont disponibles et si l'alignement avec les points
chapitrés donne un résultat exploitable — **avant** de lancer le batch ⑥ sur
plusieurs séances. Nécessite ① et ② (pas besoin de ③).

⚠️ Sous-titres **auto-générés (ASR)** : les patronymes sont parfois mal
transcrits (pas fiable pour l'attribution de parole, utile pour la recherche
de contenu) ; pas de diarisation ; langue « -orig » unique par vidéo.

In [ ]:
url = "https://raw.githubusercontent.com/pmeyssonnier/pv-explorer-app/main/pipeline/prototype_transcript.py"
code_src = urllib.request.urlopen(url).read().decode("utf-8")
exec(compile(code_src, "prototype_transcript.py", "exec"))
print("Fonctions chargées : fetch_points_for_date, fetch_transcript_segments, "
      "align_transcript_to_points, print_report")


In [ ]:
DATE = "2026-05-27"       # ← séance à tester (doit déjà avoir des points chapitrés)
VIDEO_ID = "sA80qmUc9VY"  # ← son video_id (voir backend/video_sessions.json)

seance = fetch_points_for_date(DATE)                      # noqa: F821 -- chargé ci-dessus
segments = fetch_transcript_segments(VIDEO_ID)             # noqa: F821
enriched = align_transcript_to_points(seance, segments)    # noqa: F821
print_report(enriched)                                     # noqa: F821


## ⑥ Batch transcript (plage de dates) — génère le fichier enrichi complet

Traite **toutes** les séances déjà chapitrées dont la date ≥ `MIN_DATE`,
aligne et découpe le transcript en sous-chunks d'environ 1500 caractères
(prêts pour l'indexation Pinecone — voir `index_pv.py`,
`video_point_to_chunks`). Nécessite ① et ② (pas besoin de ③ ni ⑤).

**Idempotent** : une séance déjà enrichie (run précédent) est ignorée par
défaut — relancer avec un `MIN_DATE` plus ancien pour couvrir de nouvelles
séances ne refait pas le travail déjà fait.

In [ ]:
url = "https://raw.githubusercontent.com/pmeyssonnier/pv-explorer-app/main/pipeline/extract_transcripts.py"
code_src = urllib.request.urlopen(url).read().decode("utf-8")
exec(compile(code_src, "extract_transcripts.py", "exec"))

MIN_DATE = "2000-01-01"          # ← couvre toutes les séances filmées (2020+) ; ajuste si besoin
data = run_batch(min_date=MIN_DATE)  # noqa: F821 -- chargé ci-dessus


### Et ensuite ?

Télécharge `/content/video_conseil_schaerbeek.json` et transmets-le pour
intégration (remplace `backend/video_conseil_schaerbeek.json`) — puis
réindexe Pinecone (upsert idempotent, ajoute les nouveaux chunks de
transcript sans dupliquer les chunks déjà indexés) :
```bash
python index_pv.py --commune schaerbeek --input video_conseil_schaerbeek.json
```
